# BabyWatcher — Train lại object detector (YOLOv8n, 100 epochs)

Notebook này train lại model phát hiện đối tượng dùng dataset `babyMonitor2` (Roboflow) trên
**100 epochs**, và báo cáo đầy đủ **Precision, Recall, F1-score, mAP@50, mAP@50-95** theo từng
epoch và trên tập test cuối cùng.

Dataset gốc có 4 lớp (`baby`, `blanket`, `other`, `toy`), nhưng lớp `other` chỉ có 5 nhãn
trong toàn bộ dataset — mục 2.5 tự động loại bỏ lớp này trước khi train (xem giải thích ở đó),
nên model cuối cùng nhận diện 3 lớp: `baby`, `blanket`, `toy`.

**Trước khi chạy:** vào `Runtime > Change runtime type > T4 GPU` để bật GPU miễn phí — nếu không
sẽ chạy bằng CPU và rất chậm (100 epochs có thể mất vài giờ thay vì ~20–30 phút).

**Lưu ý về tên chỉ số:** "mAP@50-90" không phải một chỉ số chuẩn trong object detection — chỉ số
chuẩn (và cũng là chỉ số Ultralytics/COCO báo cáo) là **mAP@50-95** (trung bình mAP ở các
ngưỡng IoU từ 0.50 đến 0.95, bước 0.05). Notebook dùng mAP@50-95.

## 0. Kiểm tra GPU

In [ ]:
!nvidia-smi

## 1. Cài đặt thư viện

In [ ]:
!pip install -q ultralytics roboflow

## 2. Tải dataset

### Cách A (khuyên dùng): tải trực tiếp từ Roboflow
Dataset `babyMonitor2` gốc của đồ án được export từ Roboflow project
`this-workspace-ecyl4/babymonitor2-c7by6` (xem `babyMonitor2.v1i.yolov8/data.yaml`). Lấy API key
miễn phí tại https://roboflow.com/settings/api (đăng nhập bằng tài khoản đã tạo dataset), dán vào
ô bên dưới rồi chạy.

In [ ]:
ROBOFLOW_API_KEY = ""  # @param {type:"string"}
WORKSPACE = "this-workspace-ecyl4"
PROJECT = "babymonitor2-c7by6"
VERSION = 1

assert ROBOFLOW_API_KEY, "Dán Roboflow API key vào ô ROBOFLOW_API_KEY ở trên rồi chạy lại ô này."

from roboflow import Roboflow

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace(WORKSPACE).project(PROJECT)
dataset = project.version(VERSION).download("yolov8")
DATA_YAML = f"{dataset.location}/data.yaml"
print("Dataset location:", dataset.location)
print("data.yaml:", DATA_YAML)

### Cách B (nhanh nhất): dùng sẵn `babyMonitor2_merged.zip` đã gộp + chia sẵn
Bỏ qua toàn bộ mục 2 và 2.5 nếu dùng cách này — file `babyMonitor2_merged.zip`
(được tạo bằng `merge_datasets.py` chạy local, đã gộp babyMonitor2 + 3 dataset công khai,
bỏ `other`, chia sẵn 80/10/10, `nc: 3`) được sinh sẵn ở thư mục gốc đồ án. Upload file này
lên Google Drive (thư mục gốc My Drive), rồi chạy ô dưới — **không cần Roboflow API key**.

### Cách C (dự phòng nếu không có file zip): tải lẻ từ Google Drive
Nếu chỉ có `babyMonitor2.v1i.yolov8/` gốc (chưa gộp), nén thành `babyMonitor2.zip`, upload,
rồi chạy mục 2.5 phía dưới như bình thường sau khi gán `DATA_YAML` trỏ vào file gốc này.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !unzip -q "/content/drive/MyDrive/babyMonitor2_merged.zip" -d /content
# DATA_YAML = "/content/babyMonitor2_merged.v1i.yolov8/data.yaml"
# print("data.yaml:", DATA_YAML)
# print("Da san sang huan luyen -- BO QUA muc 2.5, chay thang xuong muc 3.")

# --- Cach C (du phong, dataset chua gop) ---
# drive.mount('/content/drive')
# !unzip -q "/content/drive/MyDrive/babyMonitor2.zip" -d /content/babyMonitor2
# DATA_YAML = "/content/babyMonitor2/data.yaml"
# print("data.yaml:", DATA_YAML)

In [ ]:
import yaml

with open(DATA_YAML) as f:
    data_cfg = yaml.safe_load(f)

print("Số lớp (nc):", data_cfg.get("nc"))
print("Tên lớp:", data_cfg.get("names"))

## 2.5. Xử lý mất cân bằng dataset + gộp thêm 3 dataset công khai + chia lại 80/10/10

Dataset gốc `babyMonitor2` có 4 lớp `baby`, `blanket`, `other`, `toy`, nhưng lớp
**`other` chỉ có 5 instance trong toàn bộ 1594 ảnh** — không đủ để model học được gì
(confusion matrix của lần train trước xác nhận: recall = 0% cho lớp này). Logic phát hiện
nguy hiểm trong `src/detector.py` cũng không phân biệt `blanket`/`other`/`toy`, nên bỏ
`other` không mất khả năng phân biệt nào hệ thống thực sự dùng.

Ô dưới đây, theo thứ tự:
1. Bỏ nhãn `other`, dồn lại `toy` (3→2).
2. Tải thêm 3 dataset công khai từ Roboflow Universe (cùng workspace `nghia-thai`) để tăng
   quy mô và đa dạng dữ liệu:
   - **Baby Monitoring 4** (150 ảnh, 1 lớp `Baby`) → toàn bộ box thành `baby`.
   - **Baby-Detection** (629 ảnh, 3 lớp tư thế ngủ `prone`/`sideways`/`suspine`) → toàn bộ
     box thành `baby` (tư thế ngủ không quan trọng với bài toán này).
   - **kid-toys** (1037 ảnh, 33 loại đồ chơi hình thú/trái cây/xe cộ, định dạng
     **polygon segmentation**) → toàn bộ box thành `toy`, tự động chuyển polygon sang
     bounding box (lấy min/max toạ độ) vì đồ án train detection, không phải segmentation.
3. Gộp toàn bộ ảnh từ 4 nguồn, xáo trộn theo seed cố định (=42, khớp với
   `merge_datasets.py` chạy local), chia lại **80/10/10**.

Tỉ lệ 70/20/10 gốc của Roboflow chỉ là mặc định nền tảng, không phải lựa chọn có cơ sở
riêng cho bài toán này — xem trao đổi trong chat để biết lý do chọn 80/10/10.

In [ ]:
import random
import shutil
from pathlib import Path

from roboflow import Roboflow

SEED = 42
RATIOS = {"train": 0.8, "valid": 0.1, "test": 0.1}
NEW_NAMES = ["baby", "blanket", "toy"]


def polygon_to_bbox(coords):
    xs, ys = coords[0::2], coords[1::2]
    x_min, x_max, y_min, y_max = min(xs), max(xs), min(ys), max(ys)
    return (x_min + x_max) / 2, (y_min + y_max) / 2, x_max - x_min, y_max - y_min


def collect_pairs(dataset_dir, remap_fn, prefix):
    """remap_fn(old_cls) -> new_cls or None (drop). Handles both bbox (4
    values) and polygon-segmentation (variable-length) label lines."""
    pairs = []
    for split in ["train", "valid", "test"]:
        images_dir = Path(dataset_dir) / split / "images"
        labels_dir = Path(dataset_dir) / split / "labels"
        if not images_dir.exists():
            continue
        for img_path in images_dir.iterdir():
            if img_path.suffix.lower() not in {".jpg", ".jpeg", ".png"}:
                continue
            label_path = labels_dir / (img_path.stem + ".txt")
            lines_out = []
            if label_path.exists():
                for line in label_path.read_text(encoding="utf-8").splitlines():
                    if not line.strip():
                        continue
                    parts = line.split()
                    new_cls = remap_fn(int(parts[0]))
                    if new_cls is None:
                        continue
                    values = [float(v) for v in parts[1:]]
                    if len(values) == 4:
                        cx, cy, bw, bh = values
                    else:
                        cx, cy, bw, bh = polygon_to_bbox(values)
                    lines_out.append(f"{new_cls} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}")
            pairs.append((img_path, f"{prefix}_{img_path.name}", lines_out))
    return pairs


# --- 1) Base babyMonitor2 dataset: drop 'other' (2), remap toy 3->2 ---
BASE_REMAP = {0: 0, 1: 1, 2: None, 3: 2}
all_pairs = collect_pairs(dataset.location, lambda c: BASE_REMAP.get(c), "base")
print(f"base (babyMonitor2): {len(all_pairs)} images")

# --- 2) Download + collect the 3 supplementary public datasets ---
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

bm4 = rf.workspace("nghia-thai").project("baby-monitoring-4-smfku").version(1).download("yolov8")
bm4_pairs = collect_pairs(bm4.location, lambda c: 0, "bm4")  # single class -> baby
print(f"Baby Monitoring 4: {len(bm4_pairs)} images")
all_pairs += bm4_pairs

bd = rf.workspace("nghia-thai").project("baby-detection-wabjm-wdvut").version(1).download("yolov8")
bd_pairs = collect_pairs(bd.location, lambda c: 0, "bd")  # prone/sideways/suspine -> baby
print(f"Baby-Detection: {len(bd_pairs)} images")
all_pairs += bd_pairs

kt = rf.workspace("nghia-thai").project("kid-toys-zyngr").version(1).download("yolov8")
kt_pairs = collect_pairs(kt.location, lambda c: 2, "kt")  # all 33 toy types -> toy
print(f"kid-toys: {len(kt_pairs)} images")
all_pairs += kt_pairs

print(f"\nTotal pooled: {len(all_pairs)} images")

# --- 3) Shuffle (fixed seed) and re-split 80/10/10 ---
rng = random.Random(SEED)
rng.shuffle(all_pairs)

n = len(all_pairs)
n_train = round(n * RATIOS["train"])
n_valid = round(n * RATIOS["valid"])

split_assignment = (
    [("train", p) for p in all_pairs[:n_train]]
    + [("valid", p) for p in all_pairs[n_train:n_train + n_valid]]
    + [("test", p) for p in all_pairs[n_train + n_valid:]]
)

DST = Path(f"{dataset.location}_merged")
counts = {"train": 0, "valid": 0, "test": 0}
for split, (img_path, new_name, lines_out) in split_assignment:
    images_out, labels_out = DST / split / "images", DST / split / "labels"
    images_out.mkdir(parents=True, exist_ok=True)
    labels_out.mkdir(parents=True, exist_ok=True)
    shutil.copy2(img_path, images_out / new_name)
    (labels_out / (Path(new_name).stem + ".txt")).write_text(
        "\n".join(lines_out) + ("\n" if lines_out else ""), encoding="utf-8"
    )
    counts[split] += 1

for split in ["train", "valid", "test"]:
    print(f"{split}: {counts[split]} images ({counts[split] / n * 100:.1f}%)")

new_data_yaml = DST / "data.yaml"
new_data_yaml.write_text(
    "train: ../train/images\nval: ../valid/images\ntest: ../test/images\n\n"
    f"nc: {len(NEW_NAMES)}\nnames: {NEW_NAMES}\n",
    encoding="utf-8",
)

DATA_YAML = str(new_data_yaml)
print("\nDATA_YAML now points to:", DATA_YAML)

## 3. Huấn luyện YOLOv8n — 100 epochs

Dùng `yolov8n.pt` (YOLOv8 Nano — nhẹ, phù hợp chạy real-time trên thiết bị yếu như Jetson Nano)
làm base weights, đúng model đang được `config.yaml` của đồ án dùng làm
`object_model_path`, để trọng số mới có thể thay thế trực tiếp mà không đổi kiến trúc.

In [ ]:
from ultralytics import YOLO
from pathlib import Path

model = YOLO("yolov8n.pt")

results = model.train(
    data=DATA_YAML,
    epochs=100,
    imgsz=640,
    batch=16,
    patience=20,
    device=0,
    project="babymonitor2_runs",
    name="detector",
    exist_ok=True,
    save=True,
    plots=True,
    verbose=True,
)

# Derive RUN_DIR from the actual save directory Ultralytics used, instead of
# hardcoding "babymonitor2_runs/detector" -- the working directory can shift
# (e.g. the roboflow download step changing cwd), which would silently make a
# hardcoded path point to the wrong place even though training itself succeeded.
RUN_DIR = str(results.save_dir)
assert (Path(RUN_DIR) / "results.csv").exists(), f"results.csv not found under {RUN_DIR}"
print("Run directory:", RUN_DIR)

## 4. Precision / Recall / F1 / mAP@50 / mAP@50-95 theo từng epoch

Ultralytics ghi lại precision, recall, mAP@50, mAP@50-95 sau mỗi epoch vào `results.csv` — F1
không có sẵn theo tên đó nên được tính lại từ precision/recall (`F1 = 2PR / (P+R)`).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv(f"{RUN_DIR}/results.csv")
df.columns = [c.strip() for c in df.columns]

precision = df["metrics/precision(B)"]
recall = df["metrics/recall(B)"]
map50 = df["metrics/mAP50(B)"]
map50_95 = df["metrics/mAP50-95(B)"]
f1 = 2 * precision * recall / (precision + recall + 1e-9)
df["f1"] = f1

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(df["epoch"], precision, label="Precision")
ax.plot(df["epoch"], recall, label="Recall")
ax.plot(df["epoch"], map50, label="mAP@50")
ax.plot(df["epoch"], map50_95, label="mAP@50-95")
ax.plot(df["epoch"], f1, label="F1-score", linestyle="--")
ax.set_xlabel("Epoch")
ax.set_ylabel("Score")
ax.set_ylim(0, 1)
ax.set_title("Chỉ số huấn luyện theo epoch — YOLOv8n trên babyMonitor2 (100 epochs)")
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig("training_metrics.png", dpi=150)
plt.show()

df[["epoch", "metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)", "f1"]].tail(5)

## 5. Đánh giá cuối cùng trên tập test

Chỉ số theo epoch ở trên đo trên tập **validation** trong lúc train. Đánh giá cuối cùng chạy lại
trên tập **test** (dữ liệu chưa từng được model nhìn thấy trong lúc train/tune) để có con số báo
cáo đáng tin cậy nhất.

In [ ]:
import json

best_pt = Path(RUN_DIR) / "weights" / "best.pt"
assert best_pt.exists(), (
    f"Không tìm thấy {best_pt}. Kiểm tra lại RUN_DIR (chạy print(results.save_dir) "
    "nếu biến 'results' còn trong bộ nhớ phiên hiện tại)."
)

best_model = YOLO(str(best_pt))
test_metrics = best_model.val(data=DATA_YAML, split="test")

p = float(test_metrics.box.mp)
r = float(test_metrics.box.mr)
map50_test = float(test_metrics.box.map50)
map50_95_test = float(test_metrics.box.map)
f1_test = 2 * p * r / (p + r + 1e-9)

print("=== Kết quả trên tập TEST (best.pt) ===")
print(f"Precision:  {p:.4f}")
print(f"Recall:     {r:.4f}")
print(f"F1-score:   {f1_test:.4f}")
print(f"mAP@50:     {map50_test:.4f}")
print(f"mAP@50-95:  {map50_95_test:.4f}")

print()
print("=== Theo từng lớp ===")
per_class = {}
for i, name in test_metrics.names.items():
    cp, cr, cmap50, cmap = test_metrics.box.class_result(i)
    cf1 = 2 * cp * cr / (cp + cr + 1e-9)
    per_class[name] = {"precision": cp, "recall": cr, "f1": cf1, "map50": cmap50, "map50_95": cmap}
    print(f"{name:>10}: P={cp:.4f}  R={cr:.4f}  F1={cf1:.4f}  mAP50={cmap50:.4f}  mAP50-95={cmap:.4f}")

summary = {
    "overall": {"precision": p, "recall": r, "f1": f1_test, "map50": map50_test, "map50_95": map50_95_test},
    "per_class": per_class,
    "run_dir": str(RUN_DIR),
}
with open("test_metrics_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print()
print("Đã lưu test_metrics_summary.json")

## 6. Xem các biểu đồ Ultralytics tự sinh

Ultralytics tự vẽ `confusion_matrix.png`, `F1_curve.png`, `PR_curve.png`, `P_curve.png`,
`R_curve.png`, `results.png` (loss + metric curves) vào thư mục run khi `plots=True`.

In [ ]:
import os
from IPython.display import Image, display

for fname in ["results.png", "confusion_matrix.png", "confusion_matrix_normalized.png",
              "F1_curve.png", "PR_curve.png", "P_curve.png", "R_curve.png"]:
    path = f"{RUN_DIR}/{fname}"
    if os.path.exists(path):
        print(fname)
        display(Image(filename=path, width=720))

## 7. Lưu và tải model về

Tải `best.pt` về máy, rồi đặt vào thư mục gốc của đồ án (hoặc `models/babyMonitor2/`) và cập nhật
`config.yaml`:

```yaml
models:
  object_model_path: "models/babyMonitor2/babymonitor2_best.pt"
```

In [ ]:
import shutil
from google.colab import files

shutil.copy(f"{RUN_DIR}/weights/best.pt", "babymonitor2_best.pt")
files.download("babymonitor2_best.pt")
files.download("training_metrics.png")

### (Tuỳ chọn) Lưu toàn bộ thư mục run vào Google Drive
Giữ lại mọi biểu đồ + `results.csv` + trọng số `last.pt`/`best.pt` để đối chiếu sau này.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r babymonitor2_runs "/content/drive/MyDrive/babymonitor2_runs_50ep"
# print("Đã lưu vào Google Drive: MyDrive/babymonitor2_runs_50ep")